# Autopilota su Kaggle — motore pesante (12 GB RAM + GPU)

Esegue la piattaforma sfruttando le risorse di Kaggle, più generose di GitHub Actions: più RAM (multi-strumento / multi-timeframe), GPU opzionale, e **scheduling nativo** dei notebook.

## Setup (una volta)
1. **Add Input** → cerca e aggiungi il tuo dataset `melomac/histdata` (si monta in `/kaggle/input/histdata`).
2. (Opzionale) **Settings → Accelerator → GPU** per il clustering accelerato.
3. **Save Version → Schedule** per farlo girare da solo (es. ogni lunedì).

Gli output finiscono in `/kaggle/working` (report, EA, modelli) e diventano scaricabili / pubblicabili come dataset.

In [ ]:
# Installa la piattaforma direttamente dal repo (sempre l'ultima versione su main).
!pip install -q 'git+https://github.com/Mel0mac86/trading-ai.git@main'

In [ ]:
from pathlib import Path
from trading_ai.data_engine.sources import acquire
from trading_ai.pipeline import run_pipeline, PipelineConfig
from trading_ai.strategy_generator import RiskParams

# Il dataset montato come input (niente download: e' gia' sul disco di Kaggle).
DATA = Path('/kaggle/input/histdata')
OUT = Path('/kaggle/working')

# Fonde tutti i file annuali dello strumento in un'unica serie continua.
raw, src = acquire('XAUUSD', datasets_dir=DATA, allow_download=False)
print('Fonte:', src, '| candele:', f'{len(raw):,}',
      '|', raw.index.min().date(), '->', raw.index.max().date())

## Analisi multi-timeframe

Con 12 GB di RAM possiamo spazzolare più timeframe in un colpo solo. Per ognuno, report ed EA finiscono in `/kaggle/working`.

In [ ]:
risk = RiskParams(sl_atr=2, tp_atr=3, be_atr=1, trail_atr=1.5, max_bars=48)
summary = []
for tf in ['M15', 'H1', 'H4']:
    cfg = PipelineConfig(
        timeframe=tf, horizon=10, n_clusters=25,
        min_frequency=0.001, min_profit_factor=1.1, min_count_test=30,
        use_dsr=True, min_dsr=0.90, risk=risk, instrument=f'XAUUSD_{tf}',
        reports_dir=OUT / 'reports', mql4_dir=OUT / 'mql4', mql5_dir=OUT / 'mql5',
        validate=True, make_reports=True, export_ea=True,
    )
    res = run_pipeline(raw, cfg)
    n = len(res.robust_strategies)
    summary.append((tf, len(res.features), n))
    print(f'[{tf}] barre {len(res.features):,} | strategie robuste: {n}')

print('\nRiepilogo:', summary)
print('Output in /kaggle/working: reports/, mql4/, mql5/')

## (Opzionale) Clustering accelerato su GPU

Con GPU attiva e RAPIDS installato (`cuml`), KMeans gira su GPU: utile per alzare molto `n_clusters` e il numero di strumenti. Esempio drop-in:

```python
# from cuml.cluster import KMeans as cuKMeans  # se disponibile su Kaggle GPU
# Si puo' iniettare un clusterer GPU al posto di sklearn KMeans in FeatureClusterer.
```

## Scheduling

**Save Version → Schedule a notebook run** (giornaliero/settimanale): Kaggle rieseguirà tutto da solo, con i dati aggiornati del dataset, senza alcun intervento. Gli output restano nella cartella di lavoro della versione.